# Day 1 — Telco churn notebook
Use this notebook after `python -m src.download_data` from the project root. Open VS Code from the **project root** so `from src...` works. The full reusable code lives in `src/`; this notebook helps you understand what it does.


## 1. Load and inspect customer data
`Churn` is converted to 0/1 and blank `TotalCharges` to missing numeric values.


In [ ]:
from src.common import load_clean_data, feature_columns
import pandas as pd

df = load_clean_data()
print(df.shape)
display(df.head())
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head(10))


## 2. Inspect target imbalance
A majority-class accuracy baseline is not enough to prove a useful churn model.


In [ ]:
print(df["Churn"].value_counts())
print("Churn percentage:", round(df["Churn"].mean() * 100, 2))
display(df.groupby("Contract")["Churn"].agg(["count", "mean"]).sort_values("mean", ascending=False))


## 3. SQL analysis
The script `python -m src.build_database` creates a real SQLite database. These notebook queries are optional interactive practice.


In [ ]:
import sqlite3
from src.common import DB_PATH
# Run `python -m src.build_database` before this cell.
with sqlite3.connect(DB_PATH) as conn:
    display(pd.read_sql_query("""
        SELECT Contract, COUNT(*) AS customers,
               ROUND(100.0 * AVG(Churn), 2) AS churn_rate_pct
        FROM customers
        GROUP BY Contract
        ORDER BY churn_rate_pct DESC
    """, conn))


## 4. Split before preprocessing
Save 20% of rows as an untouched holdout. Stratification preserves the outcome proportion.


In [ ]:
from sklearn.model_selection import train_test_split
num, cat = feature_columns(df)
X = df[num + cat]
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, stratify=y, random_state=42)
print("Training:", X_train.shape, "Holdout:", X_test.shape)
print("Train churn rate:", y_train.mean(), "Test churn rate:", y_test.mean())


## 5. Cross-validation with an end-to-end pipeline
`Pipeline` ensures median imputation, scaling, and one-hot encoding fit only on training folds. Full model comparison/evaluation is implemented in `src/train.py`.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.train import create_preprocessor

pipeline = Pipeline([
    ("preprocess", create_preprocessor(num, cat)),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(pipeline, X_train, y_train, cv=folds, scoring="roc_auc")
print("Fold ROC-AUC:", auc_scores.round(3))
print("Mean CV ROC-AUC:", auc_scores.mean().round(3))


## 6. Run the full comparison
This executes Logistic Regression vs Random Forest comparison, selects based on training CV only, evaluates once on the test set, and saves the model and charts.


In [ ]:
from src.train import main as train_all
train_all()


## 7. Inspect saved metrics and predict a sample
The 0.50 threshold is illustrative, not necessarily optimal for a retention campaign.


In [ ]:
import json
from pathlib import Path
from src.common import OUTPUT
metrics = json.loads((OUTPUT / "metrics.json").read_text())
print(json.dumps(metrics, indent=2))
from src.predict_sample import main as predict_examples
predict_examples()


## Interview explanation
You selected the model by five-fold ROC-AUC on training data, reserved the test split for final evaluation, and used a pipeline to prevent preprocessing leakage. Explain that this dataset cannot estimate causal business lift or real temporal forecast accuracy.
